# 02 - Clean Hotel Booking Demand (benchmark)

**Input:** `data/hotel_bookings.csv` (raw, never modified)

**Output:** `data/processed/hotel_clean.csv`

Used only as a benchmark for day-of-week demand, nightly rate (ADR) variation,
and cancellation behavior. Steps follow Task 2 sections D4 and D5a.

In [1]:
import pandas as pd
pd.set_option("display.max_columns", None)

RAW_PATH = "../data/hotel_bookings.csv"
h = pd.read_csv(RAW_PATH)

# Stop if the raw file is not what D4 describes
assert h.shape == (119390, 32), f"Unexpected shape: {h.shape}"
print(h.shape)
print("Missing values:\n", h.isna().sum()[h.isna().sum() > 0])

(119390, 32)
Missing values:
 children         4
country        488
agent        16340
company     112593
dtype: int64


In [2]:
# children: 4 blanks - 0, then back to int (float only due to NaN)
h["children"] = h["children"].fillna(0).astype(int)

# country: 488 blanks - Unknown
h["country"] = h["country"].fillna("Unknown")

# agent: blank means no travel agent was used - 0
h["agent"] = h["agent"].fillna(0).astype(int)

# company: 94.3% empty - dropped
h = h.drop(columns=["company"])

assert h.isna().sum().sum() == 0
print("Columns remaining:", h.shape[1])

Columns remaining: 31


In [3]:
# Arrival date split across 3 columns with the month as a name
h["arrival_date"] = pd.to_datetime(
    h["arrival_date_year"].astype(str) + "-" +
    h["arrival_date_month"] + "-" +
    h["arrival_date_day_of_month"].astype(str),
    format="%Y-%B-%d")

# Real weekday (not available in salon data because of date shift)
h["weekday"] = h["arrival_date"].dt.day_name()

h["reservation_status_date"] = pd.to_datetime(h["reservation_status_date"])

print("Arrivals:", h["arrival_date"].min().date(), "to", h["arrival_date"].max().date())
print(h["weekday"].value_counts())

Arrivals: 2015-07-01 to 2017-08-31
weekday
Friday       19631
Thursday     19254
Monday       18171
Saturday     18055
Wednesday    16139
Sunday       14141
Tuesday      13999
Name: count, dtype: int64


In [4]:
print(h["reservation_status"].value_counts())   # Check outs / Cancel / No shows

h["cancel_or_noshow"] = h["reservation_status"].isin(["Canceled", "No-Show"]).astype(int)
print("Cancel/no-show rate:", round(h["cancel_or_noshow"].mean(), 3))   
print("Canceled:", (h.reservation_status == "Canceled").sum(),
      "No-show:", (h.reservation_status == "No-Show").sum())   

reservation_status
Check-Out    75166
Canceled     43017
No-Show       1207
Name: count, dtype: int64
Cancel/no-show rate: 0.37
Canceled: 43017 No-show: 1207


In [5]:
print(h["adr"].describe())
print("ADR < 0:", (h.adr < 0).sum())
print("ADR = 0:", (h.adr == 0).sum())
print("ADR > 1000:", (h.adr > 1000).sum())

# Flagged instead of drop rows stay for counts and cancellations,
# but only valid adr rows are used for rate analysis
h["adr_valid"] = (h["adr"] > 0) & (h["adr"] < 1000)
print("Rows excluded from ADR analysis:", (~h.adr_valid).sum(),
      f"({(~h.adr_valid).mean():.1%})")

count    119390.000000
mean        101.831122
std          50.535790
min          -6.380000
25%          69.290000
50%          94.575000
75%         126.000000
max        5400.000000
Name: adr, dtype: float64
ADR < 0: 1
ADR = 0: 1959
ADR > 1000: 1
Rows excluded from ADR analysis: 1961 (1.6%)


In [6]:
n_dup = h.duplicated().sum()
print(f"Exact duplicate rows: {n_dup} ({n_dup/len(h):.1%})")
# No booking ID in this dataset identical rows may be separate bookings
# reported, not removed (D4).

Exact duplicate rows: 32001 (26.8%)


In [7]:
import os
os.makedirs("../data/processed", exist_ok=True)

keep = ["hotel", "is_canceled", "reservation_status", "cancel_or_noshow",
        "lead_time", "arrival_date", "weekday",
        "stays_in_weekend_nights", "stays_in_week_nights",
        "adr", "adr_valid", "market_segment", "customer_type"]
hc = h[keep]
assert len(hc) == 119390
hc.to_csv("../data/processed/hotel_clean.csv", index=False)
print("Saved hotel_clean.csv:", hc.shape)

Saved hotel_clean.csv: (119390, 13)
